In [356]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

df = pd.read_csv("./Dataset/winequality-red.csv")

print("--- Step 1: Baseline Accuracy (Multi-Class, 11 features) ---")

X = df.drop("quality", axis=1)
y = df['quality']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

rf_baseline = RandomForestClassifier(random_state=42, n_estimators=100)
rf_baseline.fit(X_train, y_train)

y_pred_baseline = rf_baseline.predict(X_test)
acc_baseline = accuracy_score(y_test, y_pred_baseline)

print(f"1. Baseline Accuracy (Multi-Class, 6 classes): {acc_baseline:.4f}")
print("-" * 50)


# --- 3. Target Engineering (Binary Classification) ---
# Goal: Simplify the problem by classifying wine as 'Good' (Quality >= 6) or 'Bad' (Quality < 6)
print("--- Step 2: Target Engineering (Simplifying to Binary Classification) ---")

# Create a new binary target column
df['is_good'] = df['quality'].apply(lambda x: 1 if x >= 6 else 0)

# Define new features (X_binary) and new target (y_binary)
X_binary = df.drop(['quality', 'is_good'], axis=1) # Use original 11 features
y_binary = df['is_good']

X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_binary, y_binary, train_size=0.8, random_state=42
)

# Train a simple binary model (RF_binary_simple)
rf_binary_simple = RandomForestClassifier(random_state=42, n_estimators=100)
rf_binary_simple.fit(X_train_bin, y_train_bin)
acc_simple_binary = accuracy_score(y_test_bin, rf_binary_simple.predict(X_test_bin))

print(f"2. Simple Binary Accuracy (Good vs. Bad): {acc_simple_binary:.4f}")
print("   (This is the first major accuracy increase.)")
print("-" * 50)


# --- 4. Hyperparameter Tuning on Binary Model ---
print("--- Step 3: Hyperparameter Tuning (Boosting Binary Accuracy) ---")

# Define the parameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None], # None means nodes are expanded until all leaves are pure
    'min_samples_leaf': [1, 2]
}

# Setup GridSearchCV with the Random Forest model and the parameter grid
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5, # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1, # Use all available cores
    verbose=0
)

# Fit the grid search to the training data
grid_search.fit(X_train_bin, y_train_bin)

# Get the best estimator
rf_tuned = grid_search.best_estimator_

# Evaluate the tuned model
y_pred_tuned = rf_tuned.predict(X_test_bin)
acc_tuned = accuracy_score(y_test_bin, y_pred_tuned)

print(f"3. Best Parameters Found: {grid_search.best_params_}")
print(f"4. **Final Accuracy (Binary, Tuned Model): {acc_tuned:.4f}**")
print("   (This should be your highest accuracy score.)")
print("-" * 50)

# Optional: Print classification report for the tuned model
print("\nClassification Report (Tuned Binary Model):\n")
print(classification_report(y_test_bin, y_pred_tuned, target_names=['Bad Wine (0)', 'Good Wine (1)']))


--- Step 1: Baseline Accuracy (Multi-Class, 11 features) ---
1. Baseline Accuracy (Multi-Class, 6 classes): 0.6594
--------------------------------------------------
--- Step 2: Target Engineering (Simplifying to Binary Classification) ---
2. Simple Binary Accuracy (Good vs. Bad): 0.7906
   (This is the first major accuracy increase.)
--------------------------------------------------
--- Step 3: Hyperparameter Tuning (Boosting Binary Accuracy) ---
3. Best Parameters Found: {'max_depth': 20, 'min_samples_leaf': 1, 'n_estimators': 200}
4. **Final Accuracy (Binary, Tuned Model): 0.7906**
   (This should be your highest accuracy score.)
--------------------------------------------------

Classification Report (Tuned Binary Model):

               precision    recall  f1-score   support

 Bad Wine (0)       0.76      0.76      0.76       141
Good Wine (1)       0.81      0.82      0.81       179

     accuracy                           0.79       320
    macro avg       0.79      0.79     